# Problem Statement

The UCI News Aggregator dataset is a set of over 420,000 news articles that were compiled in 2014.  

Our goal for this analysis is to determine underlying trends among the articles to uncover commonalities and other links within the dataset using K-nearest neighbors, a machine learning technique that's usually used in analyzing unlabeled data.  However, in this case, it will be used to aid in further exploring the UCI News Aggregator dataset to uncover trends that we may not notice otherwise.

# Dictionary

- ID: the numeric ID of the article
- TITLE: the headline of the article
- URL: the URL of the article
- PUBLISHER: the publisher of the article
- CATEGORY: the category of the news item; one of:
    - e: entertainment
    - b: business
    - t: science and technology
    - m: health
- STORY: alphanumeric ID of the news story that the article discusses
- HOSTNAME: hostname where the article was posted
- TIMESTAMP: approximate timestamp of the article's publication, given in Unix time (seconds since midnight on Jan 1, 1970)

# Exploration

In the data exploration phase, we will cleanse the data to ensure suitable usage for modelling.

## Library Imports

We will import the NumPy, Pandas, Scikit-Learn Matplotlib, and Seaborn libraries to analyze, model and visualize our data.

Meanwhile, pickle and re will allow us to save our variables and perform internal operations within the system in which we perform this analysis.

In [1]:
# !pip install seaborn
# !pip install --upgrade pip

In [2]:
# Import the packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import re

from sklearn.feature_extraction.text import CountVectorizer

import pickle

import warnings
warnings.filterwarnings("ignore")

# Import the SentenceBERT model
from sentence_transformers import SentenceTransformer

# Classification model
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

# Post analysis
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.manifold import TSNE

# Import the 3D visualization libraries
from mpl_toolkits.mplot3d import Axes3D
import plotly.express as px
import plotly.io as pio

from sklearn.utils import resample

In [10]:
from google.colab import drive
drive.mount('/content/drive')

data_path = '/content/drive/My Drive/Online MSDS/MOD C2/Political Polarization/data/'
results_path = '/content/drive/My Drive/Online MSDS/MOD C2/Political Polarization/results/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
# Import the datasets
dataset = pd.read_csv('/content/drive/My Drive/Online MSDS/MOD C2/Political Polarization/data/MLMA_hate_speech_words_imbalanced.csv')

# News Title Classifier

Now that we've explored the data in-depth, we can move forward to analyze the data and model a classifier to predict the classification of a news article based on its title.

This is, once again, a preview of the dataset after preprocessing.

In [22]:
dataset.head()

,tweet,implicit,explicit,target_groups,annotator_sentiment,abuse_level
0,صلاة الفجر خير لك من ترديد بول البعير وسبي الن...,1,0,1,2,2
1,صراحة نفسي اشوف ولاد الوسخة اللي قالوا مدرب اج...,1,0,0,6,1
2,طيب! هي متبرجة وعبايتها ملونه وطالعة من بيتهم ...,1,0,0,0,1
3,@user @user انا اوافقك بخصوص السوريين و العراق...,0,1,1,0,1
4,هذه السعودية التي شعبها شعب الخيم و بول البعير...,1,0,1,0,1


## Univariate Modeling

Only use the title as the independent variable and the category as the dependent variable.

In [23]:
dataset.columns

Index(['tweet', 'implicit', 'explicit', 'target_groups', 'annotator_sentiment',
       'abuse_level'],
      dtype='object')

In [24]:
imbalanced = dataset
imbalanced.head()

,tweet,implicit,explicit,target_groups,annotator_sentiment,abuse_level
0,صلاة الفجر خير لك من ترديد بول البعير وسبي الن...,1,0,1,2,2
1,صراحة نفسي اشوف ولاد الوسخة اللي قالوا مدرب اج...,1,0,0,6,1
2,طيب! هي متبرجة وعبايتها ملونه وطالعة من بيتهم ...,1,0,0,0,1
3,@user @user انا اوافقك بخصوص السوريين و العراق...,0,1,1,0,1
4,هذه السعودية التي شعبها شعب الخيم و بول البعير...,1,0,1,0,1


In [25]:
imbalanced["text"] = imbalanced["tweet"]

# Preview the DataFrame
imbalanced.head()

,tweet,implicit,explicit,target_groups,annotator_sentiment,abuse_level,text
0,صلاة الفجر خير لك من ترديد بول البعير وسبي الن...,1,0,1,2,2,صلاة الفجر خير لك من ترديد بول البعير وسبي الن...
1,صراحة نفسي اشوف ولاد الوسخة اللي قالوا مدرب اج...,1,0,0,6,1,صراحة نفسي اشوف ولاد الوسخة اللي قالوا مدرب اج...
2,طيب! هي متبرجة وعبايتها ملونه وطالعة من بيتهم ...,1,0,0,0,1,طيب! هي متبرجة وعبايتها ملونه وطالعة من بيتهم ...
3,@user @user انا اوافقك بخصوص السوريين و العراق...,0,1,1,0,1,@user @user انا اوافقك بخصوص السوريين و العراق...
4,هذه السعودية التي شعبها شعب الخيم و بول البعير...,1,0,1,0,1,هذه السعودية التي شعبها شعب الخيم و بول البعير...


In [26]:
# Drop the previous columns
imbalanced = imbalanced.drop(columns=['tweet'])
imbalanced.head()

,implicit,explicit,target_groups,annotator_sentiment,abuse_level,text
0,1,0,1,2,2,صلاة الفجر خير لك من ترديد بول البعير وسبي الن...
1,1,0,0,6,1,صراحة نفسي اشوف ولاد الوسخة اللي قالوا مدرب اج...
2,1,0,0,0,1,طيب! هي متبرجة وعبايتها ملونه وطالعة من بيتهم ...
3,0,1,1,0,1,@user @user انا اوافقك بخصوص السوريين و العراق...
4,1,0,1,0,1,هذه السعودية التي شعبها شعب الخيم و بول البعير...


Apply the train/test split in preparation for modeling.

In [27]:
# X and y
X = imbalanced.drop(columns=['target_groups'])
y = imbalanced[['target_groups']]

In [28]:
X.head()

,implicit,explicit,annotator_sentiment,abuse_level,text
0,1,0,2,2,صلاة الفجر خير لك من ترديد بول البعير وسبي الن...
1,1,0,6,1,صراحة نفسي اشوف ولاد الوسخة اللي قالوا مدرب اج...
2,1,0,0,1,طيب! هي متبرجة وعبايتها ملونه وطالعة من بيتهم ...
3,0,1,0,1,@user @user انا اوافقك بخصوص السوريين و العراق...
4,1,0,0,1,هذه السعودية التي شعبها شعب الخيم و بول البعير...


In [29]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [31]:
# Train/test split
X_train_titles, X_test_titles, y_train_titles, y_test_titles = X_train['text'], X_test['text'], y_train['target_groups'], y_test['target_groups']

# Check the distribution of the dependent variable in train vs test
train_dist = X_train['text'].value_counts(normalize=True)
test_dist = X_train['text'].value_counts(normalize=True)
print("Train Distribution:")
print(train_dist)
print("\nTest Distribution:")
print(test_dist)

Train Distribution:
text
@user retard                                                                                                                     0.000335
@user faggot                                                                                                                     0.000268
@user @user retard                                                                                                               0.000268
@user nigger                                                                                                                     0.000268
@user twat                                                                                                                       0.000268
                                                                                                                                   ...   
سؤال هل شربت بول البعير @user                                                                                                    0.000067
@user rem

In [32]:
pd.Series(y_train_titles).value_counts()

,count
target_groups,
1,8671
0,3348
2,2516
3,393


## Embed Text

The sentence transformer (in this case, SentenceBERT) allows us to see the embeddings, which are vector representations of the text.

In [33]:
# Install the sentence transformers
!pip install -U sentence-transformers

Here's a function that will convert our text into embeddings to lower dimensionality.

In [34]:
# Develop a get_embeddings function to get the embeddings using the SentenceBERT model to embed the text
    # The embeddings are the vector representations of the text
def get_embeddings(text):
    """
    Get the embeddings for the text using the SentenceBERT model.
    """
    # Load the SentenceBERT model
    model = SentenceTransformer('all-MiniLM-L6-v2')

    # Get the embeddings
    embeddings = model.encode(text)

    return embeddings

In [35]:
results_path = '/content/drive/My Drive/Online MSDS/MOD C2/Political Polarization/results/'

Generate the embeddings for the title text as the dependent variables.

In [36]:
# Build merged text for each split
X_train_titles = X_train_titles.copy()
X_test_titles  = X_test_titles.copy()

# Embed directly using the content of the Series
X_train_emb = get_embeddings(X_train_titles.tolist())
X_test_emb  = get_embeddings(X_test_titles.tolist())

# Align labels to the same indices
y_train_emb = y_train_titles.reindex(X_train_titles.index)
y_test_emb  = y_test_titles.reindex(X_test_titles.index)

# Convert to numpy for sklearn
y_train_emb = y_train.to_numpy()
y_test_emb  = y_test.to_numpy()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [37]:
# Apply UMAP
import umap
reducer = umap.UMAP()
X_train_umap = reducer.fit_transform(X_train_emb)
X_test_umap  = reducer.fit_transform(X_test_emb)
y_train_umap = reducer.fit_transform(y_train_emb)
y_test_umap = reducer.fit_transform(y_test_emb)

In [38]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((14928, 5), (3733, 5), (14928, 1), (3733, 1))

In [39]:
X_train_umap.shape, X_test_umap.shape, y_train_umap.shape, y_test_umap.shape

((14928, 2), (3733, 2), (14928, 2), (3733, 2))

In [40]:
# For each entry in X_train, X_test, y_train, and y_test, remove the final element
X_train_edited = np.delete(X_train, -1, axis=1)
X_test_edited = np.delete(X_test, -1, axis=1)

In [41]:
# Append the UMAP dimensions to their respective sets
X_train_combined = np.concatenate((X_train_edited, X_train_umap), axis=1)
X_test_combined  = np.concatenate((X_test_edited, X_test_umap), axis=1)
y_train_combined = np.concatenate((y_train_emb, y_train_umap), axis=1)
y_test_combined = np.concatenate((y_test_emb, y_test_umap), axis=1)

In [42]:
X_train_combined[0]

array([1, 0, 0, 1, 6.18862771987915, 12.044242858886719], dtype=object)

In [44]:
# Organize into a dataframe with columns implicit, explicit, annotator_sentiment, abuse_level, umap_1, umap_2
X_trained_combined_df = pd.DataFrame(X_train_combined, columns=['implicit', 'explicit', 'annotator_sentiment', 'abuse_level', 'umap_1', 'umap_2'])
X_test_combined_df = pd.DataFrame(X_test_combined, columns=['implicit', 'explicit', 'annotator_sentiment', 'abuse_level', 'umap_1', 'umap_2'])

In [45]:
X_trained_combined_df.head()

,implicit,explicit,annotator_sentiment,abuse_level,umap_1,umap_2
0,1,0,0,1,6.188628,12.044243
1,1,0,4,1,11.262794,0.806005
2,0,1,4,1,4.428813,-4.184571
3,1,0,3,1,10.253608,0.246807
4,0,1,8,1,9.901342,-3.36037


In [46]:
# Combine into a single dataframe
combined_X = pd.concat([X_trained_combined_df, X_test_combined_df], axis=0)
combined_y = pd.concat([y_train, y_test], axis=0)

In [47]:
combined_X.head()

,implicit,explicit,annotator_sentiment,abuse_level,umap_1,umap_2
0,1,0,0,1,6.188628,12.044243
1,1,0,4,1,11.262794,0.806005
2,0,1,4,1,4.428813,-4.184571
3,1,0,3,1,10.253608,0.246807
4,0,1,8,1,9.901342,-3.36037


In [48]:
combined_X.shape, combined_y.shape

((18661, 6), (18661, 1))

In [49]:
# Pickle the embeddings
pickle_path = '/content/drive/My Drive/Online MSDS/MOD C2/Political Polarization/pickle/'

with open(f'{pickle_path}mlma_hate_speech_combined_imbalanced_X.pkl', 'wb') as f:
    pickle.dump(combined_X, f)
with open(f'{pickle_path}mlma_hate_speech_combined_imbalanced_y.pkl', 'wb') as f:
    pickle.dump(combined_y, f)

In [50]:
print(type(X_train_titles))

<class 'pandas.core.series.Series'>


In [51]:
# Output the titles as a csv file
X_train_titles.to_csv(f'{results_path}mlma_hate_speech_X_train_titles_combined_new_imbalanced.csv', index=False)
X_test_titles.to_csv(f'{results_path}mlma_hate_speech_X_test_titles_combined_new_imbalanced.csv', index=False)
y_train_titles.to_csv(f'{results_path}mlma_hate_speech_y_train_titles_combined_new_imbalanced.csv', index=False)
y_test_titles.to_csv(f'{results_path}mlma_hate_speech_y_test_titles_combined_new_imbalanced.csv', index=False)